<div style="border-top: 1px solid rgba(128, 128, 128, 0.45); padding-top: 10px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.65;">Walkthrough · about 2 min</span></div>

## <span style="font-family: Newsreader, Georgia, serif; font-size: 34px; font-weight: 400;">4. Training Run</span>

<div style="font-size: 19px; line-height: 1.5; max-width: 44em;">Two commands took the repository code from a one-batch check on CPU to two full 50-epoch runs on one L4 GPU, with the repository's own hyperparameters. The curves show how training behaved and which epoch each run kept.</div>

### 4.1 Commands

A debug run on CPU came first. It pushes two images through the full model, both concept losses, a backward pass and one optimizer step, then one validation and one test batch, so a broken pipeline fails before any GPU time is spent. Full training then ran as a Slurm job on one L4.

<div><div style="display: inline-block; vertical-align: top; width: 47%; margin-right: 4%; border-top: 3px solid rgba(128, 128, 128, 0.45); padding-top: 8px;"><div style="font-family: Newsreader, Georgia, serif; font-size: 22px; margin-bottom: 4px;">CPU debug check</div><div style="font-size: 15px; line-height: 1.5;"><code>python scripts/run_debug.py</code><br>Runs <code>cvit_cub.py --debug --batch_size 2 --num_workers 0</code>, which is Lightning's <code>fast_dev_run</code>: one training, one validation and one test batch. Batch size 2 avoids the upstream batch-size-1 loss bug (Section 2).</div></div><div style="display: inline-block; vertical-align: top; width: 47%; border-top: 3px solid rgba(128, 128, 128, 0.45); padding-top: 8px;"><div style="font-family: Newsreader, Georgia, serif; font-size: 22px; margin-bottom: 4px;">L4 training</div><div style="font-size: 15px; line-height: 1.5;"><code>sbatch scripts/train_l4.sbatch</code><br>Requests one L4 on <code>hpg-turin</code> with 4 CPUs and 32 GB and checks one CUDA mixed-precision step. It then runs <code>cvit_cub.py</code> for 50 epochs at batch size 32 through <code>scripts/run_training.py</code>, which adds batch timing. It fails rather than fall back to CPU.</div></div></div>

<div style="border-top: 3px solid #ac543f; padding-top: 8px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; font-weight: 600;">Live demo</span><span style="font-size: 13.5px; opacity: 0.7;"> · instant · set <code>RERUN_DEBUG = True</code> for a fresh one-batch CPU run (about 1 min, needs the cached ViT weights)</span></div>

In [ ]:
# CPU debug check: the recorded run by default, or a fresh one-batch run of the full model.
import hashlib
import subprocess

RERUN_DEBUG = False  # True: run scripts/run_debug.py now on CPU (about 1 min); rewrites training_runs/debug/
N_CLASSES = 200  # CUB-200-2011 species; an untrained classifier scores cross-entropy ln(200)


def sha256_of(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


if RERUN_DEBUG:
    assert any((PROJECT_DIR / ".cache" / "torch" / "hub" / "checkpoints").glob("*.npz")), "Prepare the cached ViT weights first."
    assert (PROJECT_DIR / "CUB-200-2011" / "CUB_200_2011").is_dir(), "run_debug.py reads CUB-200-2011/ inside PROJECT_DIR."
    subprocess.run([sys.executable, str(PROJECT_DIR / "scripts" / "run_debug.py")], cwd=str(PROJECT_DIR), check=True)
    debug = json.loads((PROJECT_DIR / "training_runs" / "debug" / "run.json").read_text())
else:
    debug = EVIDENCE["presentation_debug_evidence"]
assert debug["success"] and debug["returncode"] == 0

# The record's hashes name the code the check exercised; compare them with this checkout.
hash_checks = [
    ("cvit_cub.py", debug["entrypoint_sha256"], REPO_DIR / "cvit_cub.py"),
    ("ctc/ctc_model.py", debug["runtime_sha256"], REPO_DIR / "ctc" / "ctc_model.py"),
    ("uv.lock", debug["lock_sha256"], PROJECT_DIR / "uv.lock"),
]
same_code = ", ".join(
    "{} {}".format(name, "matches" if recorded == sha256_of(path) else "DIFFERS") for name, recorded, path in hash_checks
)
run_root = str(Path(debug["command"][0]).parents[2])  # the project copy the check ran in
metrics = debug["test_metrics"]
fact_table([
    ("Command (paths relative to the project)", " ".join(part.replace(run_root, ".") for part in debug["command"])),
    ("Host, Python", "{} · Python {}".format(debug["hostname"], debug["python"])),
    ("Lightning mode", debug["mode"]),
    ("Elapsed, including model load", "{:.1f} s · exit code {}".format(debug["elapsed_seconds"], debug["returncode"])),
    ("Code it ran vs. this checkout", same_code),
    ("Test metrics on two images", "accuracy {:.2f} · CE {:.3f} (chance: ln {} = {:.3f}) · explanation {:.3f} · concentration {:.3f}".format(
        metrics["test_acc"], metrics["test_ce_loss"], N_CLASSES, np.log(N_CLASSES),
        metrics["test_expl_loss"], metrics["test_l1_loss"])),
], title="CPU debug check · " + ("fresh run" if RERUN_DEBUG else "recorded run"))

In [ ]:
# L4 training: what Slurm recorded for both runs, and the batch script as submitted.
import hashlib
import subprocess

SHOW_FULL_SCRIPT = False  # True prints the whole batch script, not only its resource lines and launch command

script = PROJECT_DIR / "scripts" / "train_l4.sbatch"
subprocess.run(["bash", "-n", str(script)], check=True)  # syntax check only; nothing is submitted
submission = json.loads((SETUP_LOGS / "slurm-submission.json").read_text())  # the repository-split job
disjoint_job = EVIDENCE["disjoint_training_job"]  # the disjoint-split job, submitted from its run bundle
assert submission["job_id"] == REPO_RUN and disjoint_job["job_id"] == DISJOINT_RUN

stats = []
for job in (REPO_RUN, DISJOINT_RUN):
    snap = RUNS[job]["snapshot"]
    slurm = snap["scheduler"]
    assert slurm["State"] == "COMPLETED" and slurm["ExitCode"] == "0:0"
    stats.append({
        "label": "{} · Slurm elapsed".format(RUN_LABEL[job]),
        "value": slurm["Elapsed"],
        "detail": "job {} · {}, exit {} · {} epochs · {:,} batches".format(
            job, slurm["State"], slurm["ExitCode"], snap["complete_epochs"], snap["train_batches"]),
        "color": RUN_COLOR[job],
    })
stat_row(stats)

commands = [RUNS[job]["run"]["command"] for job in (REPO_RUN, DISJOINT_RUN)]
run_root = str(Path(commands[0][0]).parents[2])  # the Blue working copy the jobs ran in
runner_today = hashlib.sha256((PROJECT_DIR / "scripts" / "run_training.py").read_bytes()).hexdigest()
request = disjoint_job["resource_request"]
fact_table([
    ("Submitted, repository split", " ".join(submission["command"])),
    ("Submitted, disjoint split", " ".join(disjoint_job["command"])),
    ("Repository command inside the job", " ".join(part.replace(run_root, ".") for part in commands[0])
        + (" (same in both runs)" if commands[0] == commands[1] else " (differs between runs)")),
    ("Resources", "{} · {} · {} CPUs · {} memory · time limit {} / {}".format(
        submission["partition"], submission["gres"], submission["cpus"], submission["host_memory"],
        submission["walltime"], request["time_limit"])),
    ("Batch script vs. this checkout",
        "matches" if hashlib.sha256(script.read_bytes()).hexdigest() == submission["script_sha256"] else "DIFFERS"),
    ("Runner vs. this checkout", ", ".join(
        "{} {}".format(RUN_LABEL[job], "matches" if RUNS[job]["run"]["runner_sha256"] == runner_today else "DIFFERS")
        for job in (REPO_RUN, DISJOINT_RUN))),
], title="L4 jobs")

lines = script.read_text().splitlines()
launch = next(i for i, line in enumerate(lines) if "run_training.py" in line)
shown = lines if SHOW_FULL_SCRIPT else [line for line in lines if line.startswith("#SBATCH")] + ["..."] + lines[launch:]
print("\n".join(shown))

### 4.2 Hyperparameters

We kept every repository default, so the runs test the published configuration rather than our tuning. Two runtime settings changed. The loader uses 2 workers instead of 8 to fit the 4-CPU allocation. The repository's FP16 mixed precision now runs through PyTorch's native AMP instead of NVIDIA Apex (Section 2). The checkpoint kept for testing is the epoch with the lowest validation cross-entropy, which is the repository's own rule.

In [ ]:
# Repository defaults, read from the argparse calls in cvit_cub.py, against what each run recorded.
import ast

SETTINGS = [  # (argument, label), in the order shown
    ("learning_rate", "Learning rate"),
    ("weight_decay", "Weight decay"),
    ("warmup", "Warmup epochs, then cosine decay"),
    ("max_epochs", "Epochs"),
    ("batch_size", "Batch size"),
    ("expl_lambda", "Explanation loss weight"),
    ("attention_sparsity", "Attention concentration weight"),
    ("seed", "Seed"),
    ("num_workers", "Data-loader workers"),
]
defaults = {}
for node in ast.walk(ast.parse((REPO_DIR / "cvit_cub.py").read_text())):
    if isinstance(node, ast.Call) and getattr(node.func, "attr", "") == "add_argument":
        options = {keyword.arg: keyword.value for keyword in node.keywords}
        if "default" in options:
            defaults[node.args[0].value.lstrip("-")] = ast.literal_eval(options["default"])

assert EVIDENCE["completed_run_comparison"]["checks"]["shared_hyperparameters_equal"]
jobs = (REPO_RUN, DISJOINT_RUN)
recorded = {job: RUNS[job]["hyperparameters"] for job in jobs}
table = pd.DataFrame(
    [[defaults[key]] + [recorded[job][key] for job in jobs] for key, _ in SETTINGS],
    index=[label for _, label in SETTINGS],
    columns=["Repository default"] + [RUN_LABEL[job] for job in jobs],
).applymap("{:g}".format)
changed = [label for key, label in SETTINGS if any(recorded[job][key] != defaults[key] for job in jobs)]
display(table.style.apply(
    lambda row: ["background: rgba(190, 124, 28, 0.15)" if row.name in changed else "" for _ in row], axis=1))
rules = {RUNS[job]["snapshot"]["checkpoint_selection"] for job in jobs}
print("Changed from the defaults:", ", ".join(changed) or "nothing")
print("Checkpoint kept for testing, both runs:", " / ".join(sorted(rules)))

### 4.3 Learning curves

Both runs trained the same way. Accuracy rises slowly during the 10-epoch learning-rate warmup, fastest just after it, and levels off in the second half. Validation cross-entropy reaches its minimum mid-run and then creeps up while the training loss keeps falling. The checkpoint rule therefore keeps a mid-run epoch, not the last one.

<div style="border-top: 3px solid #ac543f; padding-top: 8px;"><span style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; font-weight: 600;">Live demo</span><span style="font-size: 13.5px; opacity: 0.7;"> · instant · change <code>EPOCH_RANGE</code>, e.g. <code>(20, 50)</code>, or drop a run from <code>SHOW_JOBS</code>, and re-run</span></div>

In [ ]:
# Figure 4.1: per-epoch curves of both runs, read from the verified TensorBoard records.
SHOW_JOBS = (REPO_RUN, DISJOINT_RUN)  # drop one to show a single run
EPOCH_RANGE = (1, 50)  # first and last epoch shown; e.g. (20, 50) to watch validation CE turn upward

PANELS = [  # (metric suffix, panel title, y-axis label, scale)
    ("acc", "Accuracy", "Accuracy (%)", 100),
    ("ce_loss", "Classification loss", "Cross-entropy (nats)", 1),
    ("expl_loss", "Concept explanation loss", "Loss", 1),
    ("l1_loss", "Attention concentration penalty", "Unweighted penalty", 1),
]
curves = {job: pd.DataFrame(RUNS[job]["epochs"]).set_index("epoch").sort_index() for job in SHOW_JOBS}
kept = {job: int(RUNS[job]["snapshot"]["best_validation_epoch"]) for job in SHOW_JOBS}
for job in SHOW_JOBS:  # the kept epoch is the validation-CE minimum, and training CE keeps falling after it
    assert kept[job] == int(curves[job].val_ce_loss.idxmin())
    assert curves[job].train_ce_loss.iloc[-1] < curves[job].train_ce_loss[kept[job]]

fig, axes = plt.subplots(2, 2, figsize=(12, 7.2), sharex=True)
for ax, (metric, title, ylabel, scale) in zip(axes.flat, PANELS):
    for job in SHOW_JOBS:
        shown = curves[job].loc[EPOCH_RANGE[0]:EPOCH_RANGE[1]]
        ax.plot(shown.index, scale * shown["train_" + metric], color=RUN_COLOR[job], linestyle="--", linewidth=1.4,
                label="{}, training".format(RUN_LABEL[job]))
        ax.plot(shown.index, scale * shown["val_" + metric], color=RUN_COLOR[job], linewidth=1.8,
                label="{}, validation".format(RUN_LABEL[job]))
        if EPOCH_RANGE[0] <= kept[job] <= EPOCH_RANGE[1]:
            ax.scatter([kept[job]], [scale * curves[job].loc[kept[job], "val_" + metric]],
                       color=RUN_COLOR[job], s=36, zorder=5)
    ax.set(title=title, ylabel=ylabel, xlim=EPOCH_RANGE)
for ax in axes[1]:
    ax.set_xlabel("Epoch")
axes[0, 0].legend(fontsize=8, loc="lower right")
fig.suptitle("Validation loss bottoms out mid-run while training keeps improving", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

warmup = RUNS[SHOW_JOBS[0]]["hyperparameters"]["warmup"]
caption(
    "<b>Figure 4.1</b> Training (dashed) and validation (solid) metrics per epoch, from the saved TensorBoard records. "
    "Dots mark the checkpoint each run kept, its lowest validation CE: "
    + "; ".join("{} at epoch {} (validation CE {:.3f}; {:.3f} at the last epoch)".format(
        RUN_LABEL[job], kept[job], curves[job].loc[kept[job], "val_ce_loss"], curves[job].val_ce_loss.iloc[-1])
        for job in SHOW_JOBS)
    + ". The first {} epochs are learning-rate warmup. The concentration penalty is the logged ".format(warmup)
    + "<code>*_l1_loss</code>, an entropy-style term, shown unweighted."
)

In [ ]:
# The kept checkpoint against the best-accuracy and last epochs, and the log lines where testing loads it.
rows = []
for job in (REPO_RUN, DISJOINT_RUN):
    curve = pd.DataFrame(RUNS[job]["epochs"]).set_index("epoch").sort_index()
    snap = RUNS[job]["snapshot"]
    kept_epoch = int(snap["best_validation_epoch"])
    assert abs(curve.loc[kept_epoch, "val_acc"] - snap["best_validation_metrics"]["val_acc"]) < 1e-9
    best = curve.val_acc.max()
    last = int(curve.index.max())
    rows.append({
        "Run": RUN_LABEL[job],
        "Kept epoch": kept_epoch,
        "Validation accuracy there (%)": 100 * curve.loc[kept_epoch, "val_acc"],
        "Best validation accuracy (%)": 100 * best,
        "Best at epoch(s)": ", ".join(str(epoch) for epoch in curve.index[curve.val_acc == best]),
        "Validation accuracy, epoch {} (%)".format(last): 100 * curve.loc[last, "val_acc"],
    })
selection = pd.DataFrame(rows).set_index("Run")
display(selection.style.format({name: "{:.2f}" for name in selection.columns if name.endswith("(%)")}))

for job in (REPO_RUN, DISJOINT_RUN):
    print("\nEnd of training, {} (job {}, stderr):".format(RUN_LABEL[job], job))
    print(RUNS[job]["stderr_completion_excerpt"].strip())
    print("Kept checkpoint:", RUNS[job]["snapshot"]["checkpoint_path"])

<div style="border-top: 3px solid #ac543f; background: rgba(172, 84, 63, 0.07); padding: 12px 18px 14px 18px;"><div style="font-size: 11px; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.65; margin-bottom: 6px;">Key point</div><div style="font-family: Newsreader, Georgia, serif; font-size: 21px; line-height: 1.4;">Nothing was tuned. Both runs use the repository's model, runner and hyperparameters unchanged, so the only difference between them is the data split from Section 3.</div></div>

<details><summary style="cursor: pointer; font-size: 15px;"><b>Backup:</b> the two failed Slurm submissions and their fixes</summary>

Both failures came from our own runner, `scripts/run_training.py`, not from the IBM repository, and both stopped before the first training batch. Neither fix touched a hyperparameter.

| Job | Error | Cause | Fix and check |
| --- | --- | --- | --- |
| `45377754` | `TypeError: on_train_batch_start() missing ... dataloader_idx` | Our timing callback required an argument that Lightning 1.9 does not pass to training hooks. | Made `dataloader_idx` optional in every hook; ran fit, validation and test through Lightning. |
| `45387301` | `FileNotFoundError: scripts/run_training.py` while hashing the runner | Python 3.8 gave a relative `__file__`, which was read after the runner changed into its run directory. | Resolved the runner path before changing directory; ran the real runner end to end with a tiny CPU model (`setup_logs/timing-callback-check.json`). |

The third submission, `45387524`, is the repository-split run above.

</details>